## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# GPU build of llama-cpp-python (Colab with a GPU runtime)
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# CPU-only alternative (generation is much slower): comment out the line above and use this one instead
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

In [ ]:
# Remaining libraries (see requirements.txt)
!pip install -q huggingface_hub pandas tiktoken pymupdf chromadb sentence-transformers langchain langchain-community langchain-chroma langchain-huggingface langchain-text-splitters

**Note**:
- After running the two cells above, restart the runtime (Google Colab) or the notebook kernel (Jupyter), then run all cells from the next one on.
- pip may print dependency-conflict warnings; they can be ignored as long as the imports below work.

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from pathlib import Path
from IPython.display import display, Markdown

# Chunking, embedding and vector database
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Downloading and loading the LLM
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [ ]:
# ---- Settings ----
USE_GPU = True    # set to False on a CPU-only runtime

# The Merck Manual PDF is not included in the repository (it is licensed for personal use).
# Upload it to the runtime, or mount Google Drive and point PDF_PATH at it, e.g.:
#   from google.colab import drive; drive.mount("/content/drive")
#   PDF_PATH = "/content/drive/MyDrive/Colab Notebooks/medical_diagnosis_manual.pdf"
PDF_PATH = "medical_diagnosis_manual.pdf"
VECTOR_DB_DIR = "medical_db"

## Question Answering using LLM

#### Downloading and Loading the model

In [ ]:
# Mistral-7B-Instruct v0.2, 6-bit quantised GGUF (about 6 GB)
model_repo_id = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_file_name = "mistral-7b-instruct-v0.2.Q6_K.gguf"

mistral_models_dir = Path.home() / "mistral_models" / "7B-Instruct-v0.2"
mistral_models_dir.mkdir(parents=True, exist_ok=True)

model_path_full = hf_hub_download(
    repo_id=model_repo_id,
    filename=model_file_name,
    local_dir=mistral_models_dir,
)
model_path_full

In [ ]:
llm = Llama(
    model_path=model_path_full,
    n_ctx=4300,                           # context window: prompt (question + retrieved chunks) + answer
    n_gpu_layers=36 if USE_GPU else 0,    # layers offloaded to the GPU
    n_batch=512,
    verbose=False,
)

#### Response

Mistral-Instruct expects every request wrapped as `[INST] ... [/INST]`. All prompts in this notebook (plain, prompt-engineered, RAG and evaluation) are built with the same helper so that they are comparable.

In [ ]:
def build_prompt(user_message, system_message=None):
    """Wrap a request in the Mistral-Instruct format. The model has no separate system role, so the system text is placed first."""
    system_part = f"{system_message}\n\n" if system_message else ""
    return f"[INST] {system_part}{user_message} [/INST]"


def llm_generate(prompt, max_tokens=256, temperature=0, top_p=0.95, top_k=50):
    """Run the model. If the answer hits max_tokens it is marked, instead of silently ending mid-sentence."""
    output = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        stop=["[INST]"],
    )
    choice = output["choices"][0]
    text = choice["text"].strip()
    if choice.get("finish_reason") == "length":
        text += " [...cut off at max_tokens]"
    return text


def show_qa(question, answer, sources=None):
    md = f"**Q:** {question}\n\n**A:** {answer}"
    if sources:
        md += f"\n\n*Sources (PDF pages): {', '.join(str(p) for p in sources)}*"
    display(Markdown(md))

In [ ]:
queries = [
    "What is the protocol for managing sepsis in a critical care unit?",
    "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",
]
for i, q in enumerate(queries, 1):
    print(f"Query {i}: {q}")

### Plain LLM (no prompt engineering, no retrieval)

In [ ]:
for q in queries:
    show_qa(q, llm_generate(build_prompt(q)))

The answers here come only from the model's training data. Check whether any answer ends with the `[...cut off at max_tokens]` marker (it means the answer was longer than the token limit), and note that nothing can be traced to a source.

## Question Answering using LLM with Prompt Engineering

In [ ]:
system_prompt = (
    "You are a medical assistant chatbot who provides information to ER physicians. "
    "Please only return the pertinent information and keep replies limited to 512 characters."
)

The chatbot assumes a base level of medical knowledge, so it does not need to explain what a condition is. It only needs to cover how to diagnose and manage it.

In [ ]:
for q in queries:
    show_qa(q, llm_generate(build_prompt(q, system_prompt)))

## Data Preparation for RAG

### Loading the Data

In [ ]:
pdf_loader = PyMuPDFLoader(PDF_PATH)
pages = pdf_loader.load()          # one Document per PDF page
print("Pages loaded:", len(pages))

### Data Cleaning

The PDF has three features that would pollute the vector database if left in:

1. **A licence notice at the bottom of every page** (the owner's email, a licence code and a "personal use only" warning). It is the same on all 4,114 pages, so it would be added to every chunk and drag unrelated chunks closer together. It also contains personal information that should not be stored in the database or printed in the notebook output.
2. **The table of contents** (lines of dots followed by page numbers). It mentions every topic, so it can outrank the real content.
3. **The alphabetical index at the back** (lists of terms with page numbers). It looks like a match for almost any keyword query.

The cell below removes the notice and drops pages that contain only the notice, table-of-contents pages and index pages. Index pages are recognised by their footer (the book name followed by a single letter instead of a chapter title). The counts are printed so they can be checked.

In [ ]:
# The licence notice that is printed at the bottom of every page
LICENSE_RE = re.compile(
    r"\s*\S+@\S+\s+[A-Z0-9]{6,}\s+"
    r"This file is meant for personal use by \S+ only\.\s+"
    r"Sharing or publishing the contents in part or full is liable for legal action\.\s*"
)
TOC_DOTS_RE = re.compile(r"\.{8,}")                 # dot leaders in the table of contents
INDEX_FOOTER_RE = re.compile(r"Edition\s+[A-Z]\s+\d+\s*$")   # "...19th Edition / Z / 4104" on index pages


def clean_pages(pages, drop_toc=True, drop_index=True):
    cleaned, dropped = [], Counter()
    for page in pages:
        text = LICENSE_RE.sub("\n", page.page_content).strip()
        if not text:
            dropped["licence-only pages"] += 1
        elif drop_toc and len(TOC_DOTS_RE.findall(text)) >= 5:
            dropped["table-of-contents pages"] += 1
        elif drop_index and INDEX_FOOTER_RE.search(text):
            dropped["index pages"] += 1
        else:
            # keep only the metadata that is needed; "page" is 1-based so it matches the PDF viewer
            cleaned.append(Document(
                page_content=text,
                metadata={"page": page.metadata["page"] + 1, "source": Path(PDF_PATH).name},
            ))
    return cleaned, dropped


clean_docs, dropped = clean_pages(pages)
print("Pages kept:", len(clean_docs))
print("Pages dropped:", dict(dropped))
print("Pages that still contain the licence notice:", sum("personal use by" in d.page_content for d in clean_docs))

### Data Overview

In [ ]:
# A page from the middle of the book, after cleaning
sample = clean_docs[len(clean_docs) // 2]
print("PDF page", sample.metadata["page"])
print(sample.page_content[:800])

### Data Chunking

In [ ]:
# chunk_size and chunk_overlap are counted in TOKENS (cl100k_base), not characters
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=512,
    chunk_overlap=20,
)
document_chunks = text_splitter.split_documents(clean_docs)   # splits the pages already in memory
print("Number of chunks:", len(document_chunks))
print("Chunks per page: %.1f" % (len(document_chunks) / len(clean_docs)))

The chunks are up to 512 tokens long (roughly 2,000 characters) with an overlap of 20 tokens. The overlap is small, so a sentence at a chunk boundary can lose some context.

In [ ]:
for i in (100, len(document_chunks) // 2, -1):
    chunk = document_chunks[i]
    print(f"--- chunk {i}, PDF page {chunk.metadata['page']} ---")
    print(chunk.page_content[:500], "\n")

### Embedding

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="thenlper/gte-large")

Turning two different chunks into vectors, and checking how similar they are.

In [ ]:
embedding_1 = embedding_model.embed_query(document_chunks[100].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[101].page_content)

print("Dimension of the embedding vector:", len(embedding_1))
print("First 5 values of vector 1:", np.round(embedding_1[:5], 4))
cosine = np.dot(embedding_1, embedding_2) / (np.linalg.norm(embedding_1) * np.linalg.norm(embedding_2))
print("Cosine similarity of neighbouring chunks:", round(float(cosine), 3))

### Vector Database

In [ ]:
# Build the database once. On later runs the saved copy is loaded instead (building twice would store every chunk twice).
vectorstore = Chroma(
    collection_name="merck_manual",
    embedding_function=embedding_model,
    persist_directory=VECTOR_DB_DIR,
)
if vectorstore._collection.count() == 0:
    vectorstore = Chroma.from_documents(
        document_chunks,
        embedding_model,
        collection_name="merck_manual",
        persist_directory=VECTOR_DB_DIR,
    )
print("Chunks in the vector database:", vectorstore._collection.count())

The embedding model is `thenlper/gte-large`, which gives 1,024-dimensional vectors. Chroma compares them with L2 distance, so a **smaller score means a closer match**.

An off-topic query is shown next to an on-topic one. The retriever always returns the k nearest chunks, even when nothing in the manual is relevant, so the scores are the only sign of a poor match.

In [ ]:
def show_search(query, k=3):
    print("QUERY:", query)
    for doc, score in vectorstore.similarity_search_with_score(query, k=k):
        snippet = doc.page_content[:150].replace("\n", " ")
        print(f"  distance {score:.3f} | PDF page {doc.metadata['page']} | {snippet}...")
    print()

show_search("How should a brain aneurysm be managed to ensure the best possible outcome?")
show_search("Black hole")

### Retriever

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

rel_docs = retriever.invoke("How should a brain aneurysm be managed to ensure the best possible outcome?")
for d in rel_docs:
    print(f"PDF page {d.metadata['page']}: {d.page_content[:300]!r}\n")

Checking that the retrieved chunks are about brain aneurysms.

### System and User Prompt Template

In [ ]:
qna_system_message = (
    "You are a medical assistant chatbot who provides information from the given context. "
    "Answer only from the given context. "
    'If the answer is not present in the context, say "I cannot answer this question as it is not in the document". '
    "The answer should maintain accuracy, but be limited to 512 characters."
)
qna_user_message_template = """Context: {context}
Question: {question}
Answer:"""

### Response Function

In [ ]:
def generate_rag_response(user_input, k=3, max_tokens=256, temperature=0, top_p=0.95, top_k=50):
    """Retrieve the k most similar chunks, build the prompt and generate an answer.
    Returns the answer and the retrieved chunks (so the sources can be shown)."""
    docs = vectorstore.similarity_search(user_input, k=k)
    context = "\n\n".join(d.page_content for d in docs)

    user_message = qna_user_message_template.format(context=context, question=user_input)
    prompt = build_prompt(user_message, qna_system_message)

    answer = llm_generate(prompt, max_tokens=max_tokens, temperature=temperature, top_p=top_p, top_k=top_k)
    return answer, docs


def ask(question, **kwargs):
    answer, docs = generate_rag_response(question, **kwargs)
    show_qa(question, answer, sources=sorted({d.metadata["page"] for d in docs}))
    return answer, docs

The function returns the retrieved chunks as well as the answer, so every answer can be shown with the PDF pages it came from. Errors are no longer caught and returned as if they were answers.

## Question Answering using RAG

In [ ]:
for q in queries:
    ask(q)

Check each answer against the pages listed under it. Two things to look for:
- whether the answer ends with the `[...cut off at max_tokens]` marker;
- whether every statement in the answer is actually in the cited pages. A small 7B model can mix up or invent details even when the retrieved text is correct.

#### A question the manual cannot answer

In [ ]:
# The system prompt tells the model to refuse when the context does not contain the answer.
off_topic = "Who won the 2018 FIFA World Cup?"
_ = ask(off_topic)

The expected answer is the refusal sentence from the system prompt. If the model answers from its own knowledge instead, the prompt alone is not a reliable guard, and a distance threshold on the retrieved chunks should be added.

### Tuning the generation parameters

These experiments change the sampling settings only (the model itself is not fine-tuned):
- `temperature` controls randomness (0 = always the most likely word);
- `top_p` and `top_k` limit the words the model may choose from;
- `max_tokens` is the length limit of the answer.

In [ ]:
experiments = [
    (queries[0], dict(temperature=0.5, max_tokens=200)),
    (queries[1], dict(temperature=0.1, max_tokens=500)),
    (queries[2], dict(top_p=0.98, top_k=20, max_tokens=256)),
]
for q, settings in experiments:
    print("Settings:", settings)
    ask(q, **settings)

With `temperature=0` the answer is the same on every run. Raising it makes answers less predictable. Compare the first experiment with the earlier answer to the same question and check each added detail against the cited pages.

## Output Evaluation

The answers are scored with an **LLM-as-a-judge** on two measures:
- **Groundedness:** is the answer derived only from the retrieved context?
- **Relevance:** does the answer address the main aspects of the question?

The judge is the same Mistral model that wrote the answers, so it is marking its own work and tends to be generous. Treat the scores as a rough check, not as proof of quality.

In [ ]:
judge_instructions = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
{metric}

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluation criteria and finish with a line of the form "Rating: <number from 1 to 5>".
"""

groundedness_rater_system_message = judge_instructions.format(
    metric="The answer should be derived only from the information presented in the context."
)
relevance_rater_system_message = judge_instructions.format(
    metric=(
        "Relevance measures how well the answer addresses the main aspects of the question, based on the context. "
        "Consider whether all and only the important aspects are contained in the answer when evaluating relevance."
    )
)

judge_user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

The relevance instructions in the original notebook asked the judge to rate the *context* rather than the *answer*. Both judges now rate the answer.

In [ ]:
def parse_rating(text):
    """Take the 1-5 rating from the judge's text. Returns None if no rating can be found."""
    matches = re.findall(r"[Rr]ating[^0-9]{0,40}([1-5])", text)
    if not matches:
        matches = re.findall(r"\b([1-5])\b", text)
    return int(matches[-1]) if matches else None


def judge_answer(question, context, answer, max_tokens=500):
    user_message = judge_user_message_template.format(question=question, context=context, answer=answer)
    grounded_text = llm_generate(build_prompt(user_message, groundedness_rater_system_message), max_tokens=max_tokens)
    relevance_text = llm_generate(build_prompt(user_message, relevance_rater_system_message), max_tokens=max_tokens)
    return grounded_text, relevance_text

The judges rate the same answers that were shown in the RAG section (the answer is generated once and then judged, not regenerated).

In [ ]:
rows, explanations = [], {}
for q in queries:
    answer, docs = generate_rag_response(q)
    context = "\n\n".join(d.page_content for d in docs)
    grounded_text, relevance_text = judge_answer(q, context, answer)
    rows.append({
        "Question": q[:70] + "...",
        "Groundedness": parse_rating(grounded_text),
        "Relevance": parse_rating(relevance_text),
    })
    explanations[q] = (grounded_text, relevance_text)

scores = pd.DataFrame(rows)
display(scores)
print("Average groundedness:", scores["Groundedness"].mean(), "| average relevance:", scores["Relevance"].mean())

The judge's explanation for one question (change the index to read the others):

In [ ]:
q = queries[0]
print("QUESTION:", q)
print("\n--- Groundedness ---\n", explanations[q][0])
print("\n--- Relevance ---\n", explanations[q][1])

Read the explanations as well as the numbers. A rating of 5 does not mean the answer is correct: the judge only compares the answer with the retrieved text and, being the same model, can miss mistakes in both.

## Actionable Insights and Business Recommendations

### Insights

1. **RAG ties answers to a source.** Each answer comes with the PDF pages it was based on, so a clinician can check it. The plain LLM answers from its training data and cannot cite anything.
2. **Retrieval quality limits answer quality.** The retriever returns the k nearest chunks even for an off-topic query, so distance scores and a refusal test are needed. Cleaning the PDF (licence footer, table of contents, index) removes content that would otherwise match many queries.
3. **Prompt format and wording matter.** The instruction wrapper, a role, a length limit and an explicit refusal rule all change the answers. Answers cut off by the token limit are now marked.
4. **Low temperature keeps answers close to the source.** Higher temperatures add wording and detail that has to be re-checked against the pages.
5. **Self-evaluation is weak.** The same model writes and grades the answers, so the groundedness and relevance scores are generous. They cannot replace review by a clinician.

### Recommendations

- **Keep a clinician in the loop and do not use this for real patient decisions.** The system is a prototype. The answers come from a 7B model and can be wrong even when the retrieved text is right.
- **Check the age of the source.** The manual in this project is the 19th edition (2011-2012). Guidance in areas such as sepsis care has changed since then, so answers drawn from it may be out of date. Use a current edition, or add current guidelines to the database.
- **Add a distance threshold** so the system says it cannot answer when no chunk is close enough, instead of relying on the prompt alone.
- **Improve retrieval:** larger overlap or section-aware chunking, hybrid keyword + vector search, and a re-ranker for the retrieved chunks.
- **Evaluate properly:** build a set of questions with answers checked by a clinician, and measure retrieval (did the right pages come back?) and answer accuracy against it, with a stronger judge model than the one being judged.
- **Consider a larger or hosted model** for better faithfulness to the retrieved text.